# 20. Patterns in Biological Data: Clustering, PCA and Machine Learning

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `expression.py`

Shared loading for Chapter 20: log2 counts per million of all sixteen


In [ ]:
"""Shared loading for Chapter 20: log2 counts per million of all sixteen
GSE52778 samples, with short treatment labels."""

import numpy as np
import pandas as pd

SHORT = {
    "Untreated": "none",
    "Dexamethasone": "dex",
    "Albuterol": "alb",
    "Albuterol_Dexamethasone": "alb+dex",
}


def load() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Samples as rows and expressed genes as columns, plus samples."""
    counts = pd.read_csv(
        "../data/GSE52778_counts.tsv.gz", sep="\t", index_col="GeneID"
    )
    samples = pd.read_csv("samples.tsv", sep="\t", index_col="sample")
    samples["label"] = samples["treatment"].map(SHORT)
    cpm = counts[samples.index] / counts[samples.index].sum() * 1e6
    expressed = cpm[cpm.mean(axis=1) >= 1]
    return np.log2(expressed + 1).T, samples

## `pca.py`

Principal components of the sixteen airway samples.


In [ ]:
# Principal components of the sixteen airway samples.
from sklearn.decomposition import PCA

from expression import load

expression, samples = load()
variable = expression.var().sort_values(ascending=False).index[:500]
centered = expression[variable] - expression[variable].mean()

pca = PCA(n_components=3)
scores = pca.fit_transform(centered)
shares = ", ".join(f"{s:.0%}" for s in pca.explained_variance_ratio_)
print(f"share of variation on PC1, PC2, PC3: {shares}")

samples[["PC1", "PC2", "PC3"]] = scores.round(1)
print(samples[["cell_line", "label", "PC1", "PC2", "PC3"]].to_string())
samples.to_csv("pca_scores.tsv", sep="\t")

## `cluster.py`

Group the samples by how alike their expression is, with no labels used.


In [ ]:
# Group the samples by how alike their expression is, with no labels used.
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

from expression import load

expression, samples = load()
variable = expression.var().sort_values(ascending=False).index[:500]

distances = pdist(expression[variable], metric="correlation")
tree = linkage(distances, method="average")
samples["cluster"] = fcluster(tree, t=2, criterion="maxclust")
print(
    samples.sort_values(["cluster", "cell_line"])[
        ["cell_line", "label", "cluster"]
    ].to_string()
)

## `classify.py`

Learn to recognize dexamethasone treated samples from expression, then ask the model about samples it has never seen.


In [ ]:
# Learn to recognize dexamethasone treated samples from expression, then
# ask the model about samples it has never seen.
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from expression import load

expression, samples = load()
treated = samples["treatment"].str.contains("Dexamethasone").astype(int)
known = samples["label"].isin(["none", "dex"])

# Every step that learns from the data sits inside the pipeline, so each
# round of cross validation repeats it using its training samples only.
model = make_pipeline(
    StandardScaler(),
    SelectKBest(f_classif, k=50),
    LogisticRegression(max_iter=1000),
)

predicted = cross_val_predict(
    model,
    expression[known],
    treated[known],
    groups=samples["cell_line"][known],
    cv=LeaveOneGroupOut(),
)
correct = (predicted == treated[known]).sum()
print(
    f"leaving out one donor at a time: {correct} of {known.sum()} right"
)

model.fit(expression[known], treated[known])
unseen = samples[~known][["cell_line", "label"]].copy()
unseen["chance_of_dex"] = model.predict_proba(expression[~known])[:, 1]
print(unseen.round(3).to_string())

## `leakage.py`

How good does a model look when the labels are random? Twice: once with gene selection inside cross validation, once with it done beforehand.


In [ ]:
# How good does a model look when the labels are random? Twice: once with
# gene selection inside cross validation, once with it done beforehand.
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from expression import load

expression, samples = load()
groups = samples["cell_line"]
labels = samples["treatment"].str.contains("Dexamethasone").to_numpy()
random = np.random.default_rng(1)
cv = LeaveOneGroupOut()

honest, leaky = [], []
for _ in range(20):
    shuffled = random.permutation(labels)  # no real signal left

    inside = make_pipeline(
        StandardScaler(),
        SelectKBest(f_classif, k=50),
        LogisticRegression(max_iter=1000),
    )
    honest.append(
        cross_val_score(
            inside, expression, shuffled, groups=groups, cv=cv
        ).mean()
    )

    # The mistake: choose genes using every sample, test samples included
    chosen = SelectKBest(f_classif, k=50).fit(expression, shuffled)
    outside = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    )
    leaky.append(
        cross_val_score(
            outside,
            chosen.transform(expression),
            shuffled,
            groups=groups,
            cv=cv,
        ).mean()
    )

print(f"random labels, genes chosen inside:  {np.mean(honest):.0%} right")
print(f"random labels, genes chosen outside: {np.mean(leaky):.0%} right")